In [ ]:
# ============================================================
# PHAT HIEN BAT THUONG REALTIME
# Deep Autoencoder + LSTM Autoencoder
# FINAL VERSION FOR THESIS
# ============================================================

import threading
import queue
import time
import joblib
import psutil

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import LabelEncoder


def load_model_compatible(model_path, compile=False):
    """
    Load .h5 model safely across Keras versions.
    Some saved .h5 files contain quantization_config in Dense layers,
    while older TensorFlow/Keras versions do not understand that field.
    """

    try:
        return load_model(
            model_path,
            compile=compile
        )

    except Exception as first_error:
        if "quantization_config" not in str(first_error):
            raise

        print(
            f"[INFO] Dang nap lai {model_path} voi che do tuong thich Keras..."
        )

        original_dense_from_config = Dense.from_config

        def dense_from_config_compatible(cls, config):
            config = dict(config)
            config.pop("quantization_config", None)
            return original_dense_from_config(config)

        Dense.from_config = classmethod(dense_from_config_compatible)

        try:
            return load_model(
                model_path,
                compile=compile
            )
        finally:
            Dense.from_config = original_dense_from_config

print("\n====================================================")
print("HE THONG GIAM SAT THOI GIAN THUC")
print("Deep Autoencoder + LSTM Autoencoder")
print("====================================================")


# ============================================================
# FILE LOG
# ============================================================

log_file = open(
    "system_alert_log.txt",
    "w",
    encoding="utf-8"
)


# ============================================================
# LOAD MODEL + SCALER
# ============================================================

print("\n[1] DANG NAP MODEL...")


try:

    traffic_model = load_model_compatible(
        "./models/deep_autoencoder_traffic.h5",
        compile=False
    )

    log_model = load_model_compatible(
        "./models/lstm_autoencoder_logs.h5",
        compile=False
    )

    traffic_scaler = joblib.load(
        "./models/traffic_scaler.pkl"
    )

    log_scaler = joblib.load(
        "./models/log_scaler.pkl"
    )

    print("[OK] Traffic Model Loaded")
    print("[OK] Log Model Loaded")
    print("[OK] Traffic Scaler Loaded")
    print("[OK] Log Scaler Loaded")

except Exception as e:

    print("\n[LOI] KHONG NAP DUOC MODEL")
    print(e)

    raise SystemExit()


# ============================================================
# 47 FEATURE TRAFFIC
# PHAI GIU DUNG THU TU LUC TRAIN
# ============================================================

TRAFFIC_FEATURES = [

    'Flow Duration',

    'Total Fwd Packets',
    'Total Backward Packets',

    'Total Length of Fwd Packets',
    'Total Length of Bwd Packets',

    'Fwd Packet Length Max',
    'Fwd Packet Length Min',
    'Fwd Packet Length Mean',
    'Fwd Packet Length Std',

    'Bwd Packet Length Max',
    'Bwd Packet Length Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Std',

    'Flow Bytes/s',
    'Flow Packets/s',

    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'Flow IAT Min',

    'Fwd IAT Total',
    'Fwd IAT Mean',
    'Fwd IAT Std',
    'Fwd IAT Max',
    'Fwd IAT Min',

    'Bwd IAT Total',
    'Bwd IAT Mean',
    'Bwd IAT Std',
    'Bwd IAT Max',
    'Bwd IAT Min',

    'Fwd PSH Flags',

    'Fwd Header Length',
    'Bwd Header Length',

    'Fwd Packets/s',
    'Bwd Packets/s',

    'Min Packet Length',
    'Max Packet Length',

    'Packet Length Mean',
    'Packet Length Std',
    'Packet Length Variance',

    'FIN Flag Count',
    'SYN Flag Count',
    'RST Flag Count',
    'PSH Flag Count',
    'ACK Flag Count',
    'URG Flag Count',

    'Average Packet Size',
    'Avg Bwd Segment Size'
]


print(
    f"[OK] Traffic Feature Count = {len(TRAFFIC_FEATURES)}"
)


# ============================================================
# QUEUE
# ============================================================

traffic_queue = queue.Queue(maxsize=500)

log_queue = queue.Queue(maxsize=500)


# ============================================================
# BIEN THONG KE
# ============================================================

traffic_alert_count = 0
log_alert_count = 0

traffic_total_count = 0
log_total_count = 0

latency_history = []

peak_ram_mb = 0

system_running = True
# ============================================================
# PHAN 2/6: DOC VA XU LY DU LIEU NETWORK TRAFFIC
# ============================================================

print("\n[2] DANG DOC VA XU LY NETWORK TRAFFIC...")


# ============================================================
# CAU HINH DATASET DEMO
# ============================================================

TRAFFIC_CSV_PATH = (
    "./dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
)

NUM_BENIGN_CALIB = 100       # Dung de tinh dynamic threshold
NUM_ATTACK_DEMO = 1900       # Dung de demo tan cong DDoS


# ============================================================
# HAM TIEN XU LY TRAFFIC
# ============================================================

def preprocess_traffic_dataframe(df_input):
    """
    Tien xu ly traffic dung chuan voi model FINAL V3:
    - Strip ten cot
    - Lay dung 47 feature
    - Xu ly inf, nan
    - Clip theo min/max cua scaler da train
    - Transform bang traffic_scaler.pkl
    """

    df = df_input.copy()

    df.columns = df.columns.str.strip()

    missing_cols = [
        col for col in TRAFFIC_FEATURES
        if col not in df.columns
    ]

    if len(missing_cols) > 0:
        raise ValueError(
            f"Thieu cot traffic: {missing_cols}"
        )

    X = df[TRAFFIC_FEATURES].copy()

    # Xu ly inf / nan
    X.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    X.fillna(
        0,
        inplace=True
    )

    # --------------------------------------------------------
    # QUAN TRONG:
    # Khong fit scaler moi.
    # Clip theo data_min_ va data_max_ cua scaler da train.
    # Day la bien the tuong duong voi viec du lieu train da cat outlier
    # truoc khi fit MinMaxScaler.
    # --------------------------------------------------------

    train_min = traffic_scaler.data_min_

    train_max = traffic_scaler.data_max_

    X_values = X.values.astype(float)

    X_values = np.clip(
        X_values,
        train_min,
        train_max
    )

    X_scaled = traffic_scaler.transform(
        X_values
    )

    return X_scaled


# ============================================================
# DOC FILE TRAFFIC
# ============================================================

try:

    df_traffic_full = pd.read_csv(
        TRAFFIC_CSV_PATH
    )

    df_traffic_full.columns = (
        df_traffic_full.columns.str.strip()
    )

    if "Label" not in df_traffic_full.columns:
        raise ValueError(
            "File traffic khong co cot Label"
        )

    # --------------------------------------------------------
    # Tao kich ban demo:
    # 100 BENIGN dau tien -> calibration
    # 1900 DDoS -> monitoring anomaly
    # --------------------------------------------------------

    df_benign = (
        df_traffic_full[
            df_traffic_full["Label"] == "BENIGN"
        ]
        .head(NUM_BENIGN_CALIB)
    )

    df_attack = (
        df_traffic_full[
            df_traffic_full["Label"] != "BENIGN"
        ]
        .head(NUM_ATTACK_DEMO)
    )

    df_traffic_stream = pd.concat(
        [
            df_benign,
            df_attack
        ],
        axis=0
    ).reset_index(drop=True)

    traffic_labels = (
        df_traffic_stream["Label"]
        .values
    )

    print(
        "-> Phan bo label Traffic:",
        df_traffic_stream["Label"]
        .value_counts()
        .to_dict()
    )

    X_real_traffic = preprocess_traffic_dataframe(
        df_traffic_stream
    )

    print(
        f"-> [OK] Traffic stream shape: {X_real_traffic.shape}"
    )

    print(
        f"-> [OK] So mau calibration BENIGN: {len(df_benign)}"
    )

    print(
        f"-> [OK] So mau attack demo: {len(df_attack)}"
    )


except Exception as e:

    print("\n[LOI] Khong xu ly duoc Traffic Dataset")
    print(e)

    log_file.close()

    raise SystemExit()


# ============================================================
# KIEM TRA INPUT MODEL TRAFFIC
# ============================================================

expected_traffic_dim = traffic_model.input_shape[-1]

actual_traffic_dim = X_real_traffic.shape[1]

if expected_traffic_dim != actual_traffic_dim:

    print("\n[LOI] Sai so chieu Traffic Model")
    print(
        f"Model can: {expected_traffic_dim} feature"
    )
    print(
        f"Du lieu co: {actual_traffic_dim} feature"
    )

    log_file.close()

    raise SystemExit()

else:

    print(
        f"-> [OK] Traffic input dim hop le: {actual_traffic_dim}"
    )
    # ============================================================
# PHAN 3/6: DOC VA XU LY DU LIEU HDFS LOG
# ============================================================

print("\n[3] DANG DOC VA XU LY HDFS LOG...")


# ============================================================
# CAU HINH DATASET LOG
# ============================================================

LOG_CSV_PATH = (
    "./dataset/HDFS_2k.log_structured.csv"
)

TIMESTEPS = 10

LOG_FEATURE_DIM = 4


# ============================================================
# HAM TIEN XU LY HDFS LOG
# ============================================================

def preprocess_hdfs_log_dataframe(df_input):
    """
    Tien xu ly HDFS Log dung chuan voi LSTM Autoencoder:
    - EventId -> EventId_num
    - Level -> Level_num
    - Component -> Component_num
    - Pid -> Pid_norm
    - Scale bang log_scaler.pkl da train
    - Dong goi thanh sequence (N, 10, 4)
    """

    df = df_input.copy()

    # --------------------------------------------------------
    # Kiem tra cot can thiet
    # --------------------------------------------------------

    required_cols = [
        "EventId",
        "Level",
        "Component",
        "Pid"
    ]

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if len(missing_cols) > 0:
        raise ValueError(
            f"Thieu cot log: {missing_cols}"
        )


    # --------------------------------------------------------
    # Feature 1: EventId -> so nguyen
    # Vi du E1 -> 1, E2 -> 2
    # --------------------------------------------------------

    df["EventId_num"] = (
        df["EventId"]
        .astype(str)
        .str.extract(r"E(\d+)")
        .astype(float)
    )


    # --------------------------------------------------------
    # Feature 2: Level -> nhi phan
    # INFO = 0, WARN = 1
    # --------------------------------------------------------

    df["Level_num"] = (
        df["Level"] == "WARN"
    ).astype(float)


    # --------------------------------------------------------
    # Feature 3: Component -> ma hoa so
    # Luu y:
    # Code train ban dau dung LabelEncoder fit tren HDFS_2k.
    # Local demo doc cung file HDFS_2k nen cach nay khop duoc.
    # Neu dung file log khac, nen luu LabelEncoder rieng.
    # --------------------------------------------------------

    le_component = LabelEncoder()

    df["Component_num"] = (
        le_component
        .fit_transform(df["Component"])
        .astype(float)
    )


    # --------------------------------------------------------
    # Feature 4: Pid normalize ve [0,1]
    # --------------------------------------------------------

    pid_min = df["Pid"].min()

    pid_max = df["Pid"].max()

    if pid_max == pid_min:
        df["Pid_norm"] = 0.0
    else:
        df["Pid_norm"] = (
            (df["Pid"] - pid_min)
            /
            (pid_max - pid_min)
        )


    # --------------------------------------------------------
    # Lay 4 feature
    # --------------------------------------------------------

    log_features = df[
        [
            "EventId_num",
            "Level_num",
            "Component_num",
            "Pid_norm"
        ]
    ].values


    # Xu ly inf / nan
    log_features = np.nan_to_num(
        log_features,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )


    # --------------------------------------------------------
    # Scale bang scaler da train
    # Khong fit scaler moi.
    # --------------------------------------------------------

    log_features_scaled = (
        log_scaler.transform(log_features)
    )


    # --------------------------------------------------------
    # Dong goi thanh sequence 10 timestep
    # --------------------------------------------------------

    n_seq = (
        len(log_features_scaled)
        // TIMESTEPS
    )

    if n_seq == 0:
        raise ValueError(
            "Khong du log de tao sequence 10 timestep"
        )

    X_log_seq = (
        log_features_scaled[
            : n_seq * TIMESTEPS
        ]
        .reshape(
            n_seq,
            TIMESTEPS,
            LOG_FEATURE_DIM
        )
    )

    return X_log_seq, n_seq


# ============================================================
# DOC VA XU LY FILE LOG
# ============================================================

try:

    df_log_full = pd.read_csv(
        LOG_CSV_PATH,
        nrows=2000
    )

    X_log_base, n_log_seq = preprocess_hdfs_log_dataframe(
        df_log_full
    )

    # --------------------------------------------------------
    # Lap lai sequence log de khop do dai voi traffic stream
    # --------------------------------------------------------

    repeat_times = int(
        np.ceil(
            len(X_real_traffic) / len(X_log_base)
        )
    )

    X_real_log = np.tile(
        X_log_base,
        (
            repeat_times,
            1,
            1
        )
    )[:len(X_real_traffic)]

    print(
        f"-> [OK] HDFS Log base shape: {X_log_base.shape}"
    )

    print(
        f"-> [OK] HDFS Log stream shape: {X_real_log.shape}"
    )

    print(
        f"-> [OK] So sequence goc: {n_log_seq}"
    )

except Exception as e:

    print("\n[LOI] Khong xu ly duoc HDFS Log")
    print(e)

    log_file.close()

    raise SystemExit()


# ============================================================
# KIEM TRA INPUT MODEL LOG
# ============================================================

expected_log_shape = log_model.input_shape[1:]

actual_log_shape = X_real_log.shape[1:]

if expected_log_shape != actual_log_shape:

    print("\n[LOI] Sai shape Log Model")
    print(
        f"Model can: {expected_log_shape}"
    )
    print(
        f"Du lieu co: {actual_log_shape}"
    )

    log_file.close()

    raise SystemExit()

else:

    print(
        f"-> [OK] Log input shape hop le: {actual_log_shape}"
    )

# ============================================================
# PHAN 4/6: TINH DYNAMIC THRESHOLD + KHOI TAO STREAMING
# ============================================================

print("\n[4] DANG TINH DYNAMIC THRESHOLD...")


# ============================================================
# 1. CALIBRATION CHO TRAFFIC
# Lay cac mau BENIGN dau tien trong luong demo de tinh nguong
# threshold = mean(MSE) + 3 * std(MSE)
# ============================================================

CALIBRATION_SIZE = 100


benign_indices = np.where(
    traffic_labels == "BENIGN"
)[0]


if len(benign_indices) == 0:
    raise ValueError(
        "Khong co mau BENIGN de tinh dynamic threshold Traffic"
    )


traffic_calib_indices = benign_indices[
    : min(CALIBRATION_SIZE, len(benign_indices))
]


X_traffic_calib = X_real_traffic[
    traffic_calib_indices
]


traffic_calib_pred = traffic_model.predict(
    X_traffic_calib,
    batch_size=512,
    verbose=0
)


traffic_calib_mse = np.mean(
    np.square(
        X_traffic_calib - traffic_calib_pred
    ),
    axis=1
)


threshold_traffic = (
    np.mean(traffic_calib_mse)
    +
    3 * np.std(traffic_calib_mse)
)


print("\n===== DYNAMIC THRESHOLD TRAFFIC =====")
print(f"So mau calibration : {len(X_traffic_calib)}")
print(f"MSE mean           : {np.mean(traffic_calib_mse):.8f}")
print(f"MSE std            : {np.std(traffic_calib_mse):.8f}")
print(f"MSE max            : {np.max(traffic_calib_mse):.8f}")
print(f"Traffic Threshold  : {threshold_traffic:.8f}")


# ============================================================
# 2. CALIBRATION CHO LOG
# Lay 100 sequence log dau tien de tinh nguong
# ============================================================

log_calib_size = min(
    CALIBRATION_SIZE,
    len(X_real_log)
)


X_log_calib = X_real_log[
    :log_calib_size
]


log_calib_pred = log_model.predict(
    X_log_calib,
    batch_size=128,
    verbose=0
)


log_calib_mse = np.mean(
    np.square(
        X_log_calib - log_calib_pred
    ),
    axis=(1, 2)
)


threshold_log = (
    np.mean(log_calib_mse)
    +
    3 * np.std(log_calib_mse)
)


print("\n===== DYNAMIC THRESHOLD LOG =====")
print(f"So sequence calibration : {len(X_log_calib)}")
print(f"MSE mean                : {np.mean(log_calib_mse):.8f}")
print(f"MSE std                 : {np.std(log_calib_mse):.8f}")
print(f"MSE max                 : {np.max(log_calib_mse):.8f}")
print(f"Log Threshold           : {threshold_log:.8f}")


# ============================================================
# 3. KHOI TAO LAI QUEUE
# ============================================================

traffic_queue = queue.Queue(
    maxsize=500
)

log_queue = queue.Queue(
    maxsize=500
)


# ============================================================
# 4. EVENT DIEU KHIEN THREAD
# ============================================================

stop_event = threading.Event()

traffic_done_event = threading.Event()

log_done_event = threading.Event()


# ============================================================
# 5. BIEN THONG KE REALTIME
# ============================================================

stats = {
    "traffic_total": 0,
    "log_total": 0,

    "traffic_alert": 0,
    "log_alert": 0,
    "hybrid_alert": 0,
    "hybrid_both_alert": 0,

    "traffic_attack_total": 0,
    "traffic_attack_detected": 0,

    "traffic_false_positive": 0,

    "latency_list": [],
    "ram_list": [],

    "start_time": None,
    "end_time": None
}


print("\n[OK] Da khoi tao Queue, Event va bien thong ke.")


# ============================================================
# 6. GHI THONG TIN THRESHOLD VAO FILE LOG
# ============================================================

log_file.write("====================================================\n")
log_file.write("SYSTEM START - DYNAMIC THRESHOLD CONFIGURATION\n")
log_file.write("====================================================\n")

log_file.write(
    f"Traffic Threshold: {threshold_traffic:.8f}\n"
)

log_file.write(
    f"Log Threshold    : {threshold_log:.8f}\n"
)

log_file.write(
    f"Traffic Calibration Samples: {len(X_traffic_calib)}\n"
)

log_file.write(
    f"Log Calibration Sequences : {len(X_log_calib)}\n"
)

log_file.write("====================================================\n\n")

log_file.flush()


print("\n[OK] Dynamic Threshold da san sang cho Streaming Realtime.")

# ============================================================
# PHAN 5/6: BA THREAD STREAMING REALTIME
# ============================================================

print("\n[5] KHOI TAO CAC THREAD STREAMING...")


# ============================================================
# CAU HINH HIEN THI DEMO
# ============================================================

TRAFFIC_SLEEP = 0.03      # Toc do stream traffic
LOG_SLEEP = 0.04          # Toc do stream log
DISPLAY_EVERY = 15        # In man hinh moi 15 mau


# Mau console
RED = "\033[91m"
GREEN = "\033[92m"
YELLOW = "\033[93m"
RESET = "\033[0m"


# ============================================================
# THREAD 1: LOG INGESTION
# ============================================================

def log_ingestion_stream():
    """
    Thread 1:
    Doc tung sequence HDFS Log va day vao log_queue.
    Moi phan tu co shape (1, 10, 4).
    """

    print("[THREAD-LOG] Bat dau stream HDFS Log...")

    for idx, log_seq in enumerate(X_real_log):

        if stop_event.is_set():
            break

        log_queue.put(
            (
                idx,
                log_seq.reshape(1, TIMESTEPS, LOG_FEATURE_DIM)
            )
        )

        time.sleep(LOG_SLEEP)

    log_done_event.set()

    print("[THREAD-LOG] Da stream het HDFS Log.")


# ============================================================
# THREAD 2: PACKET ANALYSIS / TRAFFIC STREAM
# ============================================================

def packet_analysis_stream():
    """
    Thread 2:
    Doc tung flow traffic va day vao traffic_queue.
    Moi phan tu co shape (1, 47).
    Kem theo label that de thong ke demo.
    """

    print("[THREAD-TRAFFIC] Bat dau stream Network Traffic...")

    for idx, packet in enumerate(X_real_traffic):

        if stop_event.is_set():
            break

        label = traffic_labels[idx]

        traffic_queue.put(
            (
                idx,
                packet.reshape(1, len(TRAFFIC_FEATURES)),
                label
            )
        )

        time.sleep(TRAFFIC_SLEEP)

    traffic_done_event.set()

    print("[THREAD-TRAFFIC] Da stream het Network Traffic.")


# ============================================================
# THREAD 3: HYBRID MODEL INFERENCE
# ============================================================

def hybrid_model_inference():
    """
    Thread 3:
    Lay du lieu tu traffic_queue va log_queue.
    Chay:
    - Deep Autoencoder cho traffic
    - LSTM Autoencoder cho log

    Sau do:
    - Tinh Reconstruction Error
    - So sanh voi Dynamic Threshold
    - Do Latency
    - Do RAM
    - Ghi log
    - Cap nhat thong ke
    """

    print("[THREAD-AI] AI bat dau giam sat realtime...\n")

    stats["start_time"] = time.time()

    count = 0

    while True:

        # Neu 2 producer da xong va queue da rong thi ket thuc inference
        if (
            traffic_done_event.is_set()
            and log_done_event.is_set()
            and traffic_queue.empty()
            and log_queue.empty()
        ):
            break

        # Cho den khi ca 2 queue deu co du lieu
        if traffic_queue.empty() or log_queue.empty():
            time.sleep(0.01)
            continue

        # Lay 1 mau traffic va 1 sequence log
        traffic_idx, current_packet, current_label = traffic_queue.get()
        log_idx, current_log = log_queue.get()

        start_time = time.time()

        # ====================================================
        # AI INFERENCE
        # ====================================================

        pred_traffic = traffic_model(
            current_packet,
            training=False
        ).numpy()

        pred_log = log_model(
            current_log,
            training=False
        ).numpy()


        # Reconstruction Error
        error_traffic = float(
            np.mean(
                np.square(
                    current_packet - pred_traffic
                )
            )
        )

        error_log = float(
            np.mean(
                np.square(
                    current_log - pred_log
                )
            )
        )


        # So sanh threshold dong
        traffic_is_anomaly = (
            error_traffic > threshold_traffic
        )

        log_is_anomaly = (
            error_log > threshold_log
        )


        # Hybrid decision ket hop traffic + log
        traffic_score = (
            error_traffic / threshold_traffic
            if threshold_traffic > 0
            else 0.0
        )

        log_score = (
            error_log / threshold_log
            if threshold_log > 0
            else 0.0
        )

        hybrid_score = max(
            traffic_score,
            log_score
        )

        hybrid_is_anomaly = (
            traffic_is_anomaly
            or log_is_anomaly
        )


        # Latency + RAM
        latency_ms = (
            time.time() - start_time
        ) * 1000

        ram_mb = (
            psutil.Process()
            .memory_info()
            .rss
            / (1024 * 1024)
        )


        # ====================================================
        # CAP NHAT THONG KE
        # ====================================================

        count += 1

        stats["traffic_total"] += 1
        stats["log_total"] += 1

        stats["latency_list"].append(
            latency_ms
        )

        stats["ram_list"].append(
            ram_mb
        )


        if traffic_is_anomaly:
            stats["traffic_alert"] += 1

        if log_is_anomaly:
            stats["log_alert"] += 1

        if hybrid_is_anomaly:
            stats["hybrid_alert"] += 1

        if traffic_is_anomaly and log_is_anomaly:
            stats["hybrid_both_alert"] += 1


        # Danh gia theo label that cua traffic demo
        is_attack = (
            current_label != "BENIGN"
        )

        if is_attack:
            stats["traffic_attack_total"] += 1

            if traffic_is_anomaly:
                stats["traffic_attack_detected"] += 1

        else:
            if traffic_is_anomaly:
                stats["traffic_false_positive"] += 1


        # ====================================================
        # HIEN THI CONSOLE
        # ====================================================

        traffic_status_clean = (
            "!!! CANH BAO: ANOMALY !!!"
            if traffic_is_anomaly
            else "An toan"
        )

        log_status_clean = (
            "!!! CANH BAO: ANOMALY !!!"
            if log_is_anomaly
            else "An toan"
        )

        hybrid_status_clean = (
            "!!! CANH BAO: ANOMALY !!!"
            if hybrid_is_anomaly
            else "An toan"
        )

        traffic_status_color = (
            f"{RED}{traffic_status_clean}{RESET}"
            if traffic_is_anomaly
            else f"{GREEN}{traffic_status_clean}{RESET}"
        )

        log_status_color = (
            f"{RED}{log_status_clean}{RESET}"
            if log_is_anomaly
            else f"{GREEN}{log_status_clean}{RESET}"
        )

        hybrid_status_color = (
            f"{RED}{hybrid_status_clean}{RESET}"
            if hybrid_is_anomaly
            else f"{GREEN}{hybrid_status_clean}{RESET}"
        )


        # In ra man hinh moi DISPLAY_EVERY mau
        # hoac khi co canh bao bat thuong
        if (
            count % DISPLAY_EVERY == 0
            or traffic_is_anomaly
            or log_is_anomaly
        ):

            print(
                f"================ [GIAM SAT REALTIME #{count}] ================"
            )

            print(
                f" -> Traffic Label : {current_label}"
            )

            print(
                f" -> Traffic MSE   : {error_traffic:.8f} | "
                f"{traffic_status_color}"
            )

            print(
                f" -> Sys-Log MSE   : {error_log:.8f}     | "
                f"{log_status_color}"
            )

            print(
                f" -> Hybrid Score  : {hybrid_score:.2f}       | "
                f"{hybrid_status_color}"
            )

            print(
                f" -> Hieu nang     : Latency {latency_ms:.2f} ms | "
                f"RAM {ram_mb:.2f} MB"
            )

            print(
                "==============================================================="
            )


        # ====================================================
        # GHI FILE LOG SACH, KHONG MAU
        # ====================================================

        log_file.write(
            f"#{count} | "
            f"Label={current_label} | "
            f"Traffic_MSE={error_traffic:.8f} | "
            f"Traffic_Status={traffic_status_clean} | "
            f"Log_MSE={error_log:.8f} | "
            f"Log_Status={log_status_clean} | "
            f"Hybrid_Score={hybrid_score:.4f} | "
            f"Hybrid_Status={hybrid_status_clean} | "
            f"Latency_ms={latency_ms:.2f} | "
            f"RAM_MB={ram_mb:.2f}\n"
        )

        log_file.flush()


    stats["end_time"] = time.time()

    print("\n[THREAD-AI] Da xu ly het du lieu realtime.")


    # ============================================================
# PHAN 6/6: KHOI DONG HE THONG VA IN BAO CAO TONG KET
# ============================================================

print("\n[6] KHOI DONG HE THONG STREAMING REALTIME...")


# ============================================================
# TAO THREAD
# ============================================================

thread_log = threading.Thread(
    target=log_ingestion_stream,
    name="LogIngestionThread"
)

thread_traffic = threading.Thread(
    target=packet_analysis_stream,
    name="PacketAnalysisThread"
)

thread_ai = threading.Thread(
    target=hybrid_model_inference,
    name="ModelInferenceThread"
)


# ============================================================
# START THREAD
# ============================================================

thread_log.start()
thread_traffic.start()
thread_ai.start()


# ============================================================
# DOI CAC THREAD KET THUC
# ============================================================

try:

    thread_log.join()
    thread_traffic.join()
    thread_ai.join()

except KeyboardInterrupt:

    print("\n[THONG BAO] Nguoi dung dung he thong bang Ctrl+C")

    stop_event.set()

    thread_log.join(timeout=2)
    thread_traffic.join(timeout=2)
    thread_ai.join(timeout=2)


# ============================================================
# TINH TOAN BAO CAO CUOI
# ============================================================

total_runtime = 0.0

if stats["start_time"] is not None and stats["end_time"] is not None:
    total_runtime = stats["end_time"] - stats["start_time"]


avg_latency = 0.0

if len(stats["latency_list"]) > 0:
    avg_latency = float(
        np.mean(stats["latency_list"])
    )


max_latency = 0.0

if len(stats["latency_list"]) > 0:
    max_latency = float(
        np.max(stats["latency_list"])
    )


avg_ram = 0.0

if len(stats["ram_list"]) > 0:
    avg_ram = float(
        np.mean(stats["ram_list"])
    )


peak_ram = 0.0

if len(stats["ram_list"]) > 0:
    peak_ram = float(
        np.max(stats["ram_list"])
    )


traffic_anomaly_rate = 0.0

if stats["traffic_total"] > 0:
    traffic_anomaly_rate = (
        stats["traffic_alert"]
        / stats["traffic_total"]
        * 100
    )


log_anomaly_rate = 0.0

if stats["log_total"] > 0:
    log_anomaly_rate = (
        stats["log_alert"]
        / stats["log_total"]
        * 100
    )


hybrid_anomaly_rate = 0.0

if stats["traffic_total"] > 0:
    hybrid_anomaly_rate = (
        stats["hybrid_alert"]
        / stats["traffic_total"]
        * 100
    )


traffic_detection_rate = 0.0

if stats["traffic_attack_total"] > 0:
    traffic_detection_rate = (
        stats["traffic_attack_detected"]
        / stats["traffic_attack_total"]
        * 100
    )


false_positive_rate = 0.0

benign_total = (
    stats["traffic_total"]
    - stats["traffic_attack_total"]
)

if benign_total > 0:
    false_positive_rate = (
        stats["traffic_false_positive"]
        / benign_total
        * 100
    )


# ============================================================
# IN BAO CAO CONSOLE
# ============================================================

print("\n====================================================")
print("              BAO CAO TONG KET HE THONG")
print("====================================================")

print("\n[1] TRAFFIC AUTOENCODER")
print(f"Tong so flow da xu ly        : {stats['traffic_total']}")
print(f"So flow bi canh bao anomaly  : {stats['traffic_alert']}")
print(f"Ty le anomaly Traffic        : {traffic_anomaly_rate:.2f}%")

print("\n[2] DANH GIA THEO LABEL DEMO")
print(f"So flow attack thuc te       : {stats['traffic_attack_total']}")
print(f"So attack bi phat hien       : {stats['traffic_attack_detected']}")
print(f"Detection Rate               : {traffic_detection_rate:.2f}%")
print(f"False Positive               : {stats['traffic_false_positive']}")
print(f"False Positive Rate          : {false_positive_rate:.2f}%")

print("\n[3] HDFS LOG LSTM AUTOENCODER")
print(f"Tong so log sequence xu ly   : {stats['log_total']}")
print(f"So log sequence anomaly      : {stats['log_alert']}")
print(f"Ty le anomaly Log            : {log_anomaly_rate:.2f}%")

print("\n[4] HYBRID TRAFFIC + LOG")
print(f"So canh bao Hybrid           : {stats['hybrid_alert']}")
print(f"Ty le anomaly Hybrid         : {hybrid_anomaly_rate:.2f}%")
print(f"Traffic va Log cung anomaly  : {stats['hybrid_both_alert']}")

print("\n[5] HIEU NANG REALTIME")
print(f"Thoi gian chay tong          : {total_runtime:.2f} giay")
print(f"Latency trung binh           : {avg_latency:.2f} ms")
print(f"Latency cao nhat             : {max_latency:.2f} ms")
print(f"RAM trung binh               : {avg_ram:.2f} MB")
print(f"RAM cao nhat                 : {peak_ram:.2f} MB")

print("\n[6] DYNAMIC THRESHOLD")
print(f"Traffic Threshold            : {threshold_traffic:.8f}")
print(f"Log Threshold                : {threshold_log:.8f}")

print("====================================================")
print("Log chi tiet da luu tai: system_alert_log.txt")
print("====================================================")


# ============================================================
# GHI BAO CAO CUOI VAO FILE LOG
# ============================================================

log_file.write("\n====================================================\n")
log_file.write("FINAL SYSTEM SUMMARY\n")
log_file.write("====================================================\n")

log_file.write("\n[TRAFFIC AUTOENCODER]\n")
log_file.write(f"Total Traffic Processed: {stats['traffic_total']}\n")
log_file.write(f"Traffic Alerts: {stats['traffic_alert']}\n")
log_file.write(f"Traffic Anomaly Rate: {traffic_anomaly_rate:.2f}%\n")

log_file.write("\n[DEMO LABEL EVALUATION]\n")
log_file.write(f"Actual Attack Flows: {stats['traffic_attack_total']}\n")
log_file.write(f"Detected Attack Flows: {stats['traffic_attack_detected']}\n")
log_file.write(f"Detection Rate: {traffic_detection_rate:.2f}%\n")
log_file.write(f"False Positive: {stats['traffic_false_positive']}\n")
log_file.write(f"False Positive Rate: {false_positive_rate:.2f}%\n")

log_file.write("\n[HDFS LOG LSTM AUTOENCODER]\n")
log_file.write(f"Total Log Sequences Processed: {stats['log_total']}\n")
log_file.write(f"Log Alerts: {stats['log_alert']}\n")
log_file.write(f"Log Anomaly Rate: {log_anomaly_rate:.2f}%\n")

log_file.write("\n[HYBRID TRAFFIC + LOG]\n")
log_file.write(f"Hybrid Alerts: {stats['hybrid_alert']}\n")
log_file.write(f"Hybrid Anomaly Rate: {hybrid_anomaly_rate:.2f}%\n")
log_file.write(f"Both Traffic and Log Alerts: {stats['hybrid_both_alert']}\n")

log_file.write("\n[PERFORMANCE]\n")
log_file.write(f"Total Runtime: {total_runtime:.2f} seconds\n")
log_file.write(f"Average Latency: {avg_latency:.2f} ms\n")
log_file.write(f"Max Latency: {max_latency:.2f} ms\n")
log_file.write(f"Average RAM: {avg_ram:.2f} MB\n")
log_file.write(f"Peak RAM: {peak_ram:.2f} MB\n")

log_file.write("\n[DYNAMIC THRESHOLD]\n")
log_file.write(f"Traffic Threshold: {threshold_traffic:.8f}\n")
log_file.write(f"Log Threshold: {threshold_log:.8f}\n")

log_file.write("====================================================\n")

log_file.flush()
log_file.close()


print("\n[HOAN TAT] He thong da ket thuc an toan.")